# Module 6.1: Production Pipeline Build

## Learning Objectives
By completing this notebook, you will:
1. Design production-grade data pipelines for commodity GenAI systems
2. Implement robust error handling and retry logic
3. Build data ingestion from EIA, news APIs, and other sources
4. Create end-to-end workflow orchestration

## Prerequisites
- Completion of Module 5 (Signal Generation)
- Understanding of production systems concepts
- Familiarity with APIs and data pipelines

## Estimated Time: 120 minutes

---

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
apply_course_theme()
apply_plot_theme()

In [ ]:
learning_objectives([
    "Completion of Module 5 (Signal Generation)",
    "Understanding of production systems concepts",
    "Familiarity with APIs and data pipelines"
])

## Setup & Imports

In [ ]:
# Purpose: Import libraries for production pipeline
# Key Concept: Production-grade error handling and workflow tools

import os
import json
import time
import logging
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Any, Callable
from dataclasses import dataclass, field, asdict
from enum import Enum
from pathlib import Path
import pandas as pd
import numpy as np

# Production imports
from collections import deque
from functools import wraps
import hashlib

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print("✓ Imports complete")

## 1. Production Pipeline Architecture

### Design Principles

A production commodity GenAI pipeline must be:

1. **Reliable**: Handle failures gracefully, retry transient errors
2. **Observable**: Log all operations, track metrics
3. **Scalable**: Process multiple commodities in parallel
4. **Maintainable**: Clear structure, easy to debug
5. **Cost-Efficient**: Cache results, minimize redundant LLM calls

### Pipeline Stages

```
┌─────────────────┐
│  1. INGEST      │  Fetch data from EIA, news APIs, etc.
└────────┬────────┘
         │
┌────────▼────────┐
│  2. PROCESS     │  Parse reports, extract fundamentals (LLM)
└────────┬────────┘
         │
┌────────▼────────┐
│  3. ANALYZE     │  Calculate balances, generate insights (LLM)
└────────┬────────┘
         │
┌────────▼────────┐
│  4. SIGNAL      │  Generate trading signals (LLM)
└────────┬────────┘
         │
┌────────▼────────┐
│  5. STORE       │  Save to database, cache, publish
└─────────────────┘
```

### Error Handling Strategy

- **Transient Errors**: Retry with exponential backoff (API timeouts, rate limits)
- **Permanent Errors**: Log and skip (invalid data, authentication failures)
- **Circuit Breaker**: Stop after N consecutive failures
- **Dead Letter Queue**: Store failed items for manual review

## 2. Retry and Error Handling

In [ ]:
# Purpose: Implement production-grade retry logic
# Key Concept: Resilient operations with exponential backoff

class RetryConfig:
    """Configuration for retry behavior."""
    
    def __init__(self,
                 max_attempts: int = 3,
                 initial_delay: float = 1.0,
                 max_delay: float = 60.0,
                 exponential_base: float = 2.0,
                 jitter: bool = True):
        self.max_attempts = max_attempts
        self.initial_delay = initial_delay
        self.max_delay = max_delay
        self.exponential_base = exponential_base
        self.jitter = jitter


def retry_with_backoff(config: RetryConfig = None):
    """
    Decorator for retrying functions with exponential backoff.
    
    Parameters
    ----------
    config : RetryConfig
        Retry configuration
    """
    if config is None:
        config = RetryConfig()
    
    def decorator(func: Callable) -> Callable:
        @wraps(func)
        def wrapper(*args, **kwargs):
            last_exception = None
            
            for attempt in range(config.max_attempts):
                try:
                    return func(*args, **kwargs)
                    
                except Exception as e:
                    last_exception = e
                    
                    # Don't retry on final attempt
                    if attempt == config.max_attempts - 1:
                        break
                    
                    # Calculate delay
                    delay = min(
                        config.initial_delay * (config.exponential_base ** attempt),
                        config.max_delay
                    )
                    
                    # Add jitter to prevent thundering herd
                    if config.jitter:
                        delay *= (0.5 + np.random.random())
                    
                    logger.warning(
                        f"Attempt {attempt + 1}/{config.max_attempts} failed for {func.__name__}: {e}. "
                        f"Retrying in {delay:.2f}s..."
                    )
                    
                    time.sleep(delay)
            
            # All attempts failed
            logger.error(f"All {config.max_attempts} attempts failed for {func.__name__}")
            raise last_exception
        
        return wrapper
    return decorator


class CircuitBreaker:
    """Circuit breaker pattern for failing services."""
    
    def __init__(self, 
                 failure_threshold: int = 5,
                 recovery_timeout: float = 60.0):
        """
        Initialize circuit breaker.
        
        Parameters
        ----------
        failure_threshold : int
            Number of consecutive failures before opening circuit
        recovery_timeout : float
            Seconds to wait before attempting recovery
        """
        self.failure_threshold = failure_threshold
        self.recovery_timeout = recovery_timeout
        self.failure_count = 0
        self.last_failure_time = None
        self.state = "closed"  # closed, open, half-open
    
    def call(self, func: Callable, *args, **kwargs) -> Any:
        """
        Call function through circuit breaker.
        
        Parameters
        ----------
        func : callable
            Function to call
            
        Returns
        -------
        Any
            Function result
            
        Raises
        ------
        Exception
            If circuit is open or function fails
        """
        # Check if circuit should transition to half-open
        if self.state == "open":
            if time.time() - self.last_failure_time >= self.recovery_timeout:
                self.state = "half-open"
                logger.info("Circuit breaker transitioning to half-open")
            else:
                raise Exception("Circuit breaker is OPEN - refusing to call function")
        
        try:
            result = func(*args, **kwargs)
            
            # Success - reset failure count
            if self.state == "half-open":
                logger.info("Circuit breaker recovered - transitioning to closed")
                self.state = "closed"
            
            self.failure_count = 0
            return result
            
        except Exception as e:
            self.failure_count += 1
            self.last_failure_time = time.time()
            
            # Open circuit if threshold reached
            if self.failure_count >= self.failure_threshold:
                self.state = "open"
                logger.error(
                    f"Circuit breaker OPENED after {self.failure_count} failures"
                )
            
            raise e


# Test retry logic
@retry_with_backoff(RetryConfig(max_attempts=3, initial_delay=0.1))
def flaky_api_call(fail_count: int = 2):
    """Simulate flaky API that fails N times then succeeds."""
    if not hasattr(flaky_api_call, 'attempt'):
        flaky_api_call.attempt = 0
    
    flaky_api_call.attempt += 1
    
    if flaky_api_call.attempt <= fail_count:
        raise Exception(f"API timeout (attempt {flaky_api_call.attempt})")
    
    return {"data": "success", "attempt": flaky_api_call.attempt}


# Test
result = flaky_api_call()
print(f"✓ Retry logic working: {result}")

## 3. Data Ingestion Layer

In [ ]:
# Purpose: Build data ingestion from multiple sources
# Key Concept: Unified interface for heterogeneous data sources

from abc import ABC, abstractmethod

@dataclass
class DataSource:
    """Metadata about a data source."""
    
    name: str
    source_type: str  # 'api', 'file', 'database'
    update_frequency: str  # 'daily', 'weekly', 'realtime'
    last_fetch: Optional[datetime] = None
    status: str = "active"  # 'active', 'failing', 'disabled'


class DataIngester(ABC):
    """Base class for data ingesters."""
    
    def __init__(self, source: DataSource):
        self.source = source
        self.circuit_breaker = CircuitBreaker()
        self.logger = logging.getLogger(f"{__name__}.{source.name}")
    
    @abstractmethod
    def fetch_data(self, **kwargs) -> Dict:
        """Fetch data from source."""
        pass
    
    def ingest(self, **kwargs) -> Optional[Dict]:
        """
        Ingest data with error handling.
        
        Returns
        -------
        dict or None
            Fetched data or None if failed
        """
        try:
            self.logger.info(f"Ingesting from {self.source.name}")
            
            data = self.circuit_breaker.call(self.fetch_data, **kwargs)
            
            self.source.last_fetch = datetime.now()
            self.source.status = "active"
            
            self.logger.info(f"Successfully ingested from {self.source.name}")
            return data
            
        except Exception as e:
            self.logger.error(f"Failed to ingest from {self.source.name}: {e}")
            self.source.status = "failing"
            return None


class EIAIngester(DataIngester):
    """Ingest data from EIA API."""
    
    def __init__(self, api_key: Optional[str] = None):
        source = DataSource(
            name="EIA API",
            source_type="api",
            update_frequency="weekly"
        )
        super().__init__(source)
        self.api_key = api_key or os.environ.get("EIA_API_KEY")
    
    @retry_with_backoff(RetryConfig(max_attempts=3))
    def fetch_data(self, series_id: str) -> Dict:
        """
        Fetch data from EIA API.
        
        Parameters
        ----------
        series_id : str
            EIA series ID (e.g., 'PET.WCRSTUS1.W' for crude stocks)
            
        Returns
        -------
        dict
            EIA data
        """
        # In production, this would make actual API call
        # For demo, return synthetic data
        
        self.logger.info(f"Fetching series {series_id} from EIA")
        
        # Simulate API call
        time.sleep(0.1)
        
        # Return synthetic data
        return {
            'series_id': series_id,
            'data': [
                {'date': '2023-12-22', 'value': 439.7},
                {'date': '2023-12-15', 'value': 430.8},
                {'date': '2023-12-08', 'value': 428.5}
            ],
            'units': 'million barrels',
            'fetched_at': datetime.now().isoformat()
        }


class NewsIngester(DataIngester):
    """Ingest news from RSS feeds or news APIs."""
    
    def __init__(self):
        source = DataSource(
            name="News API",
            source_type="api",
            update_frequency="realtime"
        )
        super().__init__(source)
    
    @retry_with_backoff(RetryConfig(max_attempts=2))
    def fetch_data(self, topic: str = "crude oil", hours: int = 24) -> Dict:
        """
        Fetch recent news articles.
        
        Parameters
        ----------
        topic : str
            News topic/commodity
        hours : int
            Lookback period in hours
            
        Returns
        -------
        dict
            News articles
        """
        self.logger.info(f"Fetching news for topic: {topic}")
        
        # Simulate API call
        time.sleep(0.1)
        
        return {
            'topic': topic,
            'articles': [
                {
                    'title': 'OPEC+ extends production cuts through Q1 2024',
                    'source': 'Reuters',
                    'published': (datetime.now() - timedelta(hours=2)).isoformat(),
                    'url': 'https://example.com/article1'
                },
                {
                    'title': 'US crude inventories rise unexpectedly',
                    'source': 'Bloomberg',
                    'published': (datetime.now() - timedelta(hours=5)).isoformat(),
                    'url': 'https://example.com/article2'
                }
            ],
            'fetched_at': datetime.now().isoformat()
        }


# Test ingesters
eia_ingester = EIAIngester()
eia_data = eia_ingester.ingest(series_id="PET.WCRSTUS1.W")
print(f"✓ EIA data: {eia_data['data'][0]}")

news_ingester = NewsIngester()
news_data = news_ingester.ingest(topic="crude oil")
print(f"✓ News data: {len(news_data['articles'])} articles")

## 4. Caching Layer

Reduce redundant LLM calls and API requests with intelligent caching.

In [ ]:
# Purpose: Implement caching for expensive operations
# Key Concept: Cost reduction through intelligent caching

class Cache:
    """Simple in-memory cache with TTL."""
    
    def __init__(self, default_ttl: int = 3600):
        """
        Initialize cache.
        
        Parameters
        ----------
        default_ttl : int
            Default time-to-live in seconds
        """
        self.cache = {}
        self.default_ttl = default_ttl
        self.logger = logging.getLogger(f"{__name__}.Cache")
    
    def _generate_key(self, *args, **kwargs) -> str:
        """Generate cache key from arguments."""
        key_str = json.dumps({'args': args, 'kwargs': kwargs}, sort_keys=True)
        return hashlib.md5(key_str.encode()).hexdigest()
    
    def get(self, key: str) -> Optional[Any]:
        """
        Get value from cache.
        
        Parameters
        ----------
        key : str
            Cache key
            
        Returns
        -------
        Any or None
            Cached value or None if not found/expired
        """
        if key in self.cache:
            value, expiry = self.cache[key]
            
            if datetime.now() < expiry:
                self.logger.debug(f"Cache HIT: {key}")
                return value
            else:
                # Expired
                self.logger.debug(f"Cache EXPIRED: {key}")
                del self.cache[key]
        
        self.logger.debug(f"Cache MISS: {key}")
        return None
    
    def set(self, key: str, value: Any, ttl: Optional[int] = None):
        """
        Set value in cache.
        
        Parameters
        ----------
        key : str
            Cache key
        value : Any
            Value to cache
        ttl : int, optional
            Time-to-live in seconds (uses default if None)
        """
        ttl = ttl or self.default_ttl
        expiry = datetime.now() + timedelta(seconds=ttl)
        self.cache[key] = (value, expiry)
        self.logger.debug(f"Cache SET: {key} (TTL: {ttl}s)")
    
    def cached(self, ttl: Optional[int] = None):
        """
        Decorator for caching function results.
        
        Parameters
        ----------
        ttl : int, optional
            Time-to-live in seconds
        """
        def decorator(func: Callable) -> Callable:
            @wraps(func)
            def wrapper(*args, **kwargs):
                # Generate cache key
                key = f"{func.__name__}:{self._generate_key(*args, **kwargs)}"
                
                # Check cache
                cached_value = self.get(key)
                if cached_value is not None:
                    return cached_value
                
                # Execute function
                result = func(*args, **kwargs)
                
                # Cache result
                self.set(key, result, ttl)
                
                return result
            
            return wrapper
        return decorator
    
    def clear(self):
        """Clear all cached entries."""
        self.cache.clear()
        self.logger.info("Cache cleared")
    
    def stats(self) -> Dict:
        """Get cache statistics."""
        total = len(self.cache)
        expired = sum(1 for _, expiry in self.cache.values() if datetime.now() >= expiry)
        
        return {
            'total_entries': total,
            'active_entries': total - expired,
            'expired_entries': expired
        }


# Initialize cache
cache = Cache(default_ttl=3600)  # 1 hour default

# Example: Cache expensive LLM extraction
@cache.cached(ttl=7200)  # Cache for 2 hours
def expensive_llm_extraction(report_text: str) -> Dict:
    """Simulate expensive LLM call."""
    logger.info("Executing expensive LLM extraction...")
    time.sleep(0.1)  # Simulate latency
    return {'extracted_data': 'fundamentals', 'cost': 0.05}

# Test caching
report = "Sample EIA report text..."

print("First call (cache miss):")
result1 = expensive_llm_extraction(report)
print(f"  Result: {result1}")

print("\nSecond call (cache hit):")
result2 = expensive_llm_extraction(report)
print(f"  Result: {result2}")

print(f"\nCache stats: {cache.stats()}")

## 5. Pipeline Orchestration

In [ ]:
# Purpose: Build end-to-end pipeline orchestration
# Key Concept: Coordinating multi-stage workflows

@dataclass
class PipelineRun:
    """Metadata about a pipeline run."""
    
    run_id: str
    start_time: datetime
    end_time: Optional[datetime] = None
    status: str = "running"  # running, completed, failed
    stages_completed: List[str] = field(default_factory=list)
    errors: List[str] = field(default_factory=list)
    metrics: Dict = field(default_factory=dict)


class ProductionPipeline:
    """Production-grade commodity analysis pipeline."""
    
    def __init__(self, 
                 commodities: List[str],
                 cache: Cache):
        """
        Initialize pipeline.
        
        Parameters
        ----------
        commodities : list of str
            Commodities to process
        cache : Cache
            Cache instance
        """
        self.commodities = commodities
        self.cache = cache
        self.logger = logging.getLogger(f"{__name__}.Pipeline")
        
        # Initialize ingesters
        self.eia_ingester = EIAIngester()
        self.news_ingester = NewsIngester()
        
        # Pipeline state
        self.current_run: Optional[PipelineRun] = None
    
    def run(self) -> PipelineRun:
        """
        Execute full pipeline.
        
        Returns
        -------
        PipelineRun
            Pipeline run metadata
        """
        # Initialize run
        run_id = datetime.now().strftime('%Y%m%d_%H%M%S')
        self.current_run = PipelineRun(
            run_id=run_id,
            start_time=datetime.now()
        )
        
        self.logger.info(f"Starting pipeline run: {run_id}")
        
        try:
            # Stage 1: Ingest data
            ingested_data = self._stage_ingest()
            self.current_run.stages_completed.append("ingest")
            
            # Stage 2: Process data
            processed_data = self._stage_process(ingested_data)
            self.current_run.stages_completed.append("process")
            
            # Stage 3: Analyze fundamentals
            analysis = self._stage_analyze(processed_data)
            self.current_run.stages_completed.append("analyze")
            
            # Stage 4: Generate signals
            signals = self._stage_signal(analysis)
            self.current_run.stages_completed.append("signal")
            
            # Stage 5: Store results
            self._stage_store(signals)
            self.current_run.stages_completed.append("store")
            
            # Mark as completed
            self.current_run.status = "completed"
            self.current_run.end_time = datetime.now()
            
            duration = (self.current_run.end_time - self.current_run.start_time).total_seconds()
            self.logger.info(f"Pipeline completed successfully in {duration:.2f}s")
            
        except Exception as e:
            self.current_run.status = "failed"
            self.current_run.end_time = datetime.now()
            self.current_run.errors.append(str(e))
            self.logger.error(f"Pipeline failed: {e}")
            raise
        
        return self.current_run
    
    def _stage_ingest(self) -> Dict:
        """Stage 1: Ingest data from sources."""
        self.logger.info("Stage 1: Ingesting data")
        
        ingested = {}
        
        for commodity in self.commodities:
            try:
                # Fetch EIA data
                eia_data = self.eia_ingester.ingest(
                    series_id=f"PET.{commodity}.W"
                )
                
                # Fetch news
                news_data = self.news_ingester.ingest(
                    topic=commodity
                )
                
                ingested[commodity] = {
                    'eia': eia_data,
                    'news': news_data
                }
                
            except Exception as e:
                self.logger.error(f"Failed to ingest {commodity}: {e}")
                self.current_run.errors.append(f"Ingest error for {commodity}: {e}")
        
        self.current_run.metrics['commodities_ingested'] = len(ingested)
        return ingested
    
    def _stage_process(self, ingested_data: Dict) -> Dict:
        """Stage 2: Process and parse data."""
        self.logger.info("Stage 2: Processing data")
        
        processed = {}
        
        for commodity, data in ingested_data.items():
            processed[commodity] = {
                'fundamentals': self._parse_eia_data(data['eia']),
                'sentiment': self._parse_news_sentiment(data['news'])
            }
        
        return processed
    
    def _stage_analyze(self, processed_data: Dict) -> Dict:
        """Stage 3: Analyze fundamentals."""
        self.logger.info("Stage 3: Analyzing fundamentals")
        
        analysis = {}
        
        for commodity, data in processed_data.items():
            analysis[commodity] = {
                'balance': self._calculate_balance(data['fundamentals']),
                'outlook': self._generate_outlook(data)
            }
        
        return analysis
    
    def _stage_signal(self, analysis: Dict) -> Dict:
        """Stage 4: Generate trading signals."""
        self.logger.info("Stage 4: Generating signals")
        
        signals = {}
        
        for commodity, data in analysis.items():
            signals[commodity] = self._generate_signal(commodity, data)
        
        self.current_run.metrics['signals_generated'] = len(signals)
        return signals
    
    def _stage_store(self, signals: Dict):
        """Stage 5: Store results."""
        self.logger.info("Stage 5: Storing results")
        
        # In production: save to database, publish to message queue, etc.
        # For demo: just log
        
        for commodity, signal in signals.items():
            self.logger.info(f"Stored signal for {commodity}: {signal}")
    
    # Simplified processing methods (would use actual LLMs in production)
    
    def _parse_eia_data(self, eia_data: Dict) -> Dict:
        """Parse EIA data (simplified)."""
        if not eia_data:
            return {}
        return {
            'inventory': eia_data['data'][0]['value'],
            'weekly_change': eia_data['data'][0]['value'] - eia_data['data'][1]['value']
        }
    
    def _parse_news_sentiment(self, news_data: Dict) -> Dict:
        """Parse news sentiment (simplified)."""
        if not news_data:
            return {'score': 0}
        return {'score': 0.5, 'article_count': len(news_data.get('articles', []))}
    
    def _calculate_balance(self, fundamentals: Dict) -> str:
        """Calculate fundamental balance (simplified)."""
        return "balanced"
    
    def _generate_outlook(self, data: Dict) -> str:
        """Generate outlook (simplified)."""
        return "neutral"
    
    def _generate_signal(self, commodity: str, analysis: Dict) -> Dict:
        """Generate signal (simplified)."""
        return {
            'commodity': commodity,
            'direction': 'neutral',
            'confidence': 0.6,
            'generated_at': datetime.now().isoformat()
        }


# Test pipeline
pipeline = ProductionPipeline(
    commodities=["WTI Crude", "Natural Gas"],
    cache=cache
)

run_result = pipeline.run()

print("\n=== PIPELINE RUN COMPLETE ===")
print(f"Run ID: {run_result.run_id}")
print(f"Status: {run_result.status}")
print(f"Duration: {(run_result.end_time - run_result.start_time).total_seconds():.2f}s")
print(f"Stages: {', '.join(run_result.stages_completed)}")
print(f"Metrics: {run_result.metrics}")
if run_result.errors:
    print(f"Errors: {run_result.errors}")

## 6. Scheduling and Automation

In production, pipelines run on schedules (e.g., after EIA releases, hourly for news).

In [ ]:
# Purpose: Demonstrate scheduling patterns
# Key Concept: Automated pipeline execution

from typing import Callable
from threading import Timer

class Scheduler:
    """Simple scheduler for pipeline runs."""
    
    def __init__(self):
        self.jobs = []
        self.logger = logging.getLogger(f"{__name__}.Scheduler")
    
    def schedule_daily(self, 
                      hour: int, 
                      minute: int,
                      task: Callable,
                      name: str = "unnamed"):
        """
        Schedule task to run daily at specific time.
        
        Parameters
        ----------
        hour : int
            Hour (0-23)
        minute : int
            Minute (0-59)
        task : callable
            Function to execute
        name : str
            Job name
        """
        self.logger.info(f"Scheduled {name} for daily at {hour:02d}:{minute:02d}")
        self.jobs.append({
            'name': name,
            'schedule': f"daily {hour:02d}:{minute:02d}",
            'task': task
        })
    
    def schedule_interval(self,
                         interval_minutes: int,
                         task: Callable,
                         name: str = "unnamed"):
        """
        Schedule task to run at regular intervals.
        
        Parameters
        ----------
        interval_minutes : int
            Interval in minutes
        task : callable
            Function to execute
        name : str
            Job name
        """
        self.logger.info(f"Scheduled {name} for every {interval_minutes} minutes")
        self.jobs.append({
            'name': name,
            'schedule': f"every {interval_minutes}m",
            'task': task
        })
    
    def list_jobs(self) -> pd.DataFrame:
        """List all scheduled jobs."""
        return pd.DataFrame(self.jobs)[['name', 'schedule']]


# Example usage
scheduler = Scheduler()

# Schedule pipeline runs
scheduler.schedule_daily(
    hour=11,  # 11 AM (after EIA report at 10:30 AM ET)
    minute=0,
    task=pipeline.run,
    name="EIA Weekly Pipeline"
)

scheduler.schedule_interval(
    interval_minutes=60,
    task=lambda: news_ingester.ingest(topic="crude oil"),
    name="Hourly News Scan"
)

print("=== SCHEDULED JOBS ===")
print(scheduler.list_jobs())

print("\nIn production, use:")
print("  - Airflow for complex DAG workflows")
print("  - Cron for simple schedules")
print("  - Cloud Functions/Lambda for serverless")
print("  - APScheduler for Python-based scheduling")

## Exercise 6.1: Build Your Own Pipeline Stage

In [ ]:
# Exercise: Add a new data source to the pipeline
#
# Task:
# 1. Create a new DataIngester subclass for weather data (NOAA)
# 2. Add weather ingestion to the pipeline
# 3. Incorporate weather into fundamental analysis
# 4. Run the enhanced pipeline

# YOUR CODE HERE
# ---------------

# Step 1: Create WeatherIngester
class WeatherIngester(DataIngester):
    pass  # Implement this

# Step 2: Modify pipeline to include weather
# ...

# Step 3: Update analysis logic
# ...

# Step 4: Run pipeline
# enhanced_pipeline.run()

## Summary

### Key Takeaways

1. **Resilience**: Production pipelines need retry logic, circuit breakers, and error handling
2. **Caching**: Reduce costs by caching expensive LLM calls and API responses
3. **Modularity**: Break pipelines into discrete stages for easier debugging
4. **Observability**: Log all operations for troubleshooting
5. **Automation**: Schedule pipelines to run automatically on data releases

### What's Next

In **Module 6.2: Monitoring Setup**, we'll:
- Build monitoring dashboards
- Implement alerting for failures
- Track pipeline performance metrics
- Monitor LLM quality over time

### Additional Resources

- [Apache Airflow](https://airflow.apache.org/) - Workflow orchestration
- Circuit Breaker pattern: Martin Fowler's blog
- "Designing Data-Intensive Applications" by Martin Kleppmann
- Twelve-Factor App methodology

---

**Practice Exercise**: Deploy this pipeline to run weekly after EIA reports. Monitor success rates and latency over 4 weeks.